# Exercise 2 — Industrial transfer learning with MVTec Capsule — Solution

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
repo_root = Path.cwd().parents[1] if Path.cwd().name == "solutions" else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(repo_root / "src"))
from cvis_ml.config import DatasetConfig
from cvis_ml.data import MVTecCapsuleDataModule
from cvis_ml.models import TransferModelFactory, count_parameters, describe_trainable_parameters
from cvis_ml.engine import Trainer
from cvis_ml.visualization import show_batch, plot_history, show_confusion_matrix
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
mvtec_cfg = DatasetConfig(
    name="mvtec_capsule_teaching_split",
    data_root=str(repo_root / "data_cache"),
    image_size=128,
    batch_size=16,
    max_train_samples=160,
    max_val_samples=80,
    num_workers=2,
    seed=42,
)
mvtec_dm = MVTecCapsuleDataModule(
    root=mvtec_cfg.data_root,
    image_size=mvtec_cfg.image_size,
    batch_size=mvtec_cfg.batch_size,
    num_workers=mvtec_cfg.num_workers,
    max_train_samples=mvtec_cfg.max_train_samples,
    max_val_samples=mvtec_cfg.max_val_samples,
    seed=mvtec_cfg.seed,
    augment=True,
)
mvtec_data = mvtec_dm.setup()
print("Classes:", mvtec_data.class_names)
print("Number of classes:", mvtec_data.num_classes)
print("Train batches:", len(mvtec_data.train_loader))
print("Validation batches:", len(mvtec_data.val_loader))
show_batch(mvtec_data.train_loader, mvtec_data.class_names, n=8)

In [ ]:
architecture = "mobilenet_v3_small"
strategy = "frozen"
industrial_model = TransferModelFactory.create(architecture, mvtec_data.num_classes, strategy, pretrained=True)
total, trainable = count_parameters(industrial_model)
print("Total parameters:", total)
print("Trainable parameters:", trainable)
for row in describe_trainable_parameters(industrial_model):
    print(row)

In [ ]:
learning_rate = 1e-3 if strategy == "frozen" else 1e-4
industrial_trainer = Trainer(industrial_model, device="auto", learning_rate=learning_rate, weight_decay=1e-4)
industrial_result = industrial_trainer.fit(
    train_loader=mvtec_data.train_loader,
    val_loader=mvtec_data.val_loader,
    epochs=1,
    max_batches_per_epoch=None,
    name=f"{architecture}_{strategy}_mvtec_capsule",
)
plot_history(industrial_result.history, title="Industrial baseline")
show_confusion_matrix(industrial_result.y_true, industrial_result.y_pred, mvtec_data.class_names, title="Industrial baseline")

In [ ]:
industrial_interpretation = """
A one-epoch CPU baseline should be interpreted cautiously. The key outputs are not only accuracy but also macro-F1 and the confusion matrix.
If the abnormal class has lower recall, the model may miss defects, which would be industrially critical.
The teaching split is not the official anomaly-detection protocol, so the result should not be presented as a benchmark result.
Augmentation should be checked for operational plausibility: small rotations and mild brightness changes may be plausible, but aggressive transforms may create unrealistic samples.
This is a useful baseline experiment, but not an industrially deployable model.
"""
print(industrial_interpretation)

## Optional comparison loop

In [ ]:
experiments = [
    {"name": "mobilenet_v3_small_frozen", "architecture": "mobilenet_v3_small", "strategy": "frozen"},
    {"name": "resnet18_frozen", "architecture": "resnet18", "strategy": "frozen"},
    {"name": "resnet18_partial", "architecture": "resnet18", "strategy": "partial"},
]
rows = []
for exp in experiments:
    print("\nRunning", exp)
    m = TransferModelFactory.create(exp["architecture"], mvtec_data.num_classes, exp["strategy"], pretrained=True)
    total, trainable = count_parameters(m)
    lr = 1e-3 if exp["strategy"] == "frozen" else 1e-4
    tr = Trainer(m, device="auto", learning_rate=lr, weight_decay=1e-4)
    r = tr.fit(mvtec_data.train_loader, mvtec_data.val_loader, epochs=1, max_batches_per_epoch=None, name=exp["name"])
    rows.append({"experiment": exp["name"], "total_params": total, "trainable_params": trainable,
                 "val_accuracy": r.metrics["accuracy"], "val_macro_f1": r.metrics["macro_f1"], "runtime_s": r.metrics["runtime_s"]})
pd.DataFrame(rows)